In [ ]:
import datetime
import glob
import matplotlib.pyplot as plt
import numpy as np
import pathlib
import time
import torch
import torch.optim as optim

import neuralpde


nc_files = [pathlib.Path(p) for p in sorted(glob.glob('data/V6/*.nc'))]
npz_files = [pathlib.Path(p) for p in sorted(glob.glob('data/V6/*interp.npz'))]

d = neuralpde.nc.SeaIceV6([nc_files[1]])
npz = np.load(npz_files[1])


Initialize some stuff:

In [ ]:
U0 = 1.0000e+00
L0 = 2.5000e+04
T0 = 1.0644e+03
K0 = 1.0000e+03
V0 = 5.0000e-02
F0 = 9.3949e-04

s = 24 * 60 * 60 / T0  # day in seconds
h = 25e3 / L0

n = neuralpde.network.SeaIceAdr_x5_5k3_t3_w128_d3(
    U0,
    L0,
    T0,
    K0,
    V0,
    F0,
    s = s,
    h = h
).to('cuda')


Some test cases:

In [ ]:
idx0_for = np.s_[100:103, 207:218, 227:238]
idx1_for = np.s_[100:103, 45:56, 144:155]
idx2_for = np.s_[200:203, 80:91, 185:196]

n.forward(
    torch.from_numpy(
            np.stack(
            (
                d.seaice_conc[idx0_for],
                d.seaice_conc[idx1_for],
                d.seaice_conc[idx2_for],
            )
        )
    ).to('cuda', dtype=neuralpde.network.DTYPE)
)


Test forward from npz:

In [ ]:
idx0_for = np.s_[100:103, 207:218, 227:238]
idx1_for = np.s_[100:103, 45:56, 144:155]
idx2_for = np.s_[200:203, 80:91, 185:196]

n.forward(
    torch.from_numpy(
            np.stack(
            (
                npz['seaice_conc_interp'][idx0_for],
                npz['seaice_conc_interp'][idx1_for],
                npz['seaice_conc_interp'][idx2_for],
            )
        )
    ).to('cuda', dtype=neuralpde.network.DTYPE)
)


In [ ]:
idx0_fit = np.s_[100:103, 207:220, 227:240]
idx1_fit = np.s_[100:103, 45:58, 144:157]
idx2_fit = np.s_[200:203, 80:93, 185:198]

idx0_fitl = np.s_[4, 213, 233]
idx1_fitl = np.s_[4, 51, 150]
idx2_fitl = np.s_[4, 86, 191]

n.fit(
    torch.from_numpy(
        np.stack(
            (
                d.seaice_conc[idx0_fit],
                d.seaice_conc[idx1_fit],
                d.seaice_conc[idx2_fit],
            )
        )
    ).to('cuda', dtype=neuralpde.network.DTYPE),
    torch.from_numpy(
        np.stack(
            (
                d.seaice_conc[idx0_fitl],
                d.seaice_conc[idx1_fitl],
                d.seaice_conc[idx2_fitl],
            )
        )
    ).to('cuda', dtype=neuralpde.network.DTYPE)
)


In [ ]:
BATCHSIZE = 20000
EPOCHS = 100

C = 3
N, M = (13, 13)
HX, HY = N // 2, M // 2

ld = len(d.date)
lx = len(d.x)
ly = len(d.y)


In [ ]:
losses = []
optimizer = optim.Adam(n.parameters(), lr=1e-2)

data = np.empty((BATCHSIZE, C, N, M), dtype=np.float32)
label = np.empty((BATCHSIZE,), dtype=np.float32)

n.train()

try:
    for e in range(EPOCHS):
        for es in range(ld * lx * ly // BATCHSIZE):  # approximation of steps per epoch with BATCHSIZE, discounting possible nan
            if es % 50 == 0:
                f = npz_files[np.random.randint(len(npz_files))]
                print(f'\nShuffling (changing to file {f.name})...')
                npz = np.load(f)
                seaice_conc = npz['seaice_conc_interp'].copy()
                valid = (~npz['mask']).copy()
                valid[:C - 1] = False
                valid[-1:] = False
                valid[:, :HX] = False
                valid[:, -HX:] = False
                valid[:, :, :HY] = False
                valid[:, :, -HY:] = False
                n_valid = np.sum(valid)
                it, ix, iy = np.nonzero(valid)

            tb = time.perf_counter()
            for ii, jj in enumerate(np.random.randint(n_valid, size=BATCHSIZE)):
                data[ii] = seaice_conc[it[jj] - C + 1:it[jj] + 1, ix[jj] - HX:ix[jj] + HX + 1, iy[jj] - HY:iy[jj] + HY + 1]
                label[ii] = seaice_conc[it[jj] + 1, ix[jj], iy[jj]]

            tt = time.perf_counter()
            data_t = torch.as_tensor(data, dtype=neuralpde.network.DTYPE, device='cuda')
            label_t = torch.as_tensor(label, dtype=neuralpde.network.DTYPE, device='cuda')

            optimizer.zero_grad(set_to_none=True)
            loss = n.fit(data_t, label_t).mean()
            if torch.isnan(loss):
                print(f'\nEpoch {e + 1:4} (batch {es + 1:4}):\tLoss is NaN!  Skipping regression...')
            else:
                loss.backward()
                optimizer.step()

            losses.append(loss.item())
            tf = time.perf_counter()
            print(f'Epoch {e + 1:4} (batch {es + 1:4}):\tLoss: {loss.item():10e}\t\tBatching time: {tt - tb:.3f} seconds\t\tTraining time: {tf - tt:.3f} seconds', end='\r')

    dest = f'{datetime.datetime.now().strftime(r"%Y%m%d%H%M%S")}-epoch{e + 1:04d}-batch{es + 1:04d}-SeaIceAdr_x5_5k3_t3_w128_d3.pth'
    n.save(dest)

except KeyboardInterrupt:
        dest = f'{datetime.datetime.now().strftime(r"%Y%m%d%H%M%S")}-epoch{e + 1:04d}-batch{es + 1:04d}-SeaIceAdr_x5_5k3_t3_w128_d3.pth'
        print(f'Caught keyboard interrupt.  Saving to {dest}...')
        n.save(dest)


In [ ]:
if False:
    n.load(sorted(glob.glob('*SeaIceAdr_x5_5k3_t3_w128_d3.pth'))[-1])


We can compute the solution across a range of coordinates:

In [ ]:
cmap_conc = plt.get_cmap('Blues_r')
cmap_conc.set_bad("#ff7057", alpha=0.6)

cmap_err = plt.get_cmap('bwr')
cmap_err.set_bad("#ff7057", alpha=0.6)

cmap_other = plt.get_cmap('viridis')
cmap_other.set_bad("#ff7057", alpha=0.6)


def plot_outputs(outputs_np: np.ndarray, truth: np.ndarray, date_as_str: str):
    fig, (ax0, ax1, ax2, ax3, ax4) = plt.subplots(1, 5, figsize=(24, 5), dpi=300)

    m = ax0.imshow(
        outputs_np[..., 0].T,
        cmap=cmap_conc,
        vmin=0.,
        vmax=1.
    )
    plt.colorbar(m, ax=ax0, label='Fractional Sea Ice Coverage')

    m = ax1.imshow(
        (truth - outputs_np[..., 0]).T,
        cmap=cmap_err,
        vmin=-1,
        vmax=1.
    )
    plt.colorbar(m, ax=ax1, label='Truth - Prediction')

    m = ax2.imshow(
        K0 * outputs_np[..., 1].T / 1e3 ** 2 * 60 * 60,
        cmap=cmap_other,
        vmin=-20.,
        vmax=20.,
    )
    plt.colorbar(m, ax=ax2, label='Diffusivity (km$^2$ / hr)')

    m = ax3.imshow(
        V0 * np.sqrt(np.sum((outputs_np[..., 2:4]) ** 2, axis=-1)).T / 1e3 * 60 * 60,
        cmap=cmap_other,
        vmin=0.,
        vmax=3.,
    )
    plt.colorbar(m, ax=ax3, label='Velocity (km / hr)')

    m = ax4.imshow(
        F0 * outputs_np[..., 4].T * 24 * 60 * 60,
        cmap=cmap_other,
        vmin=-2.,
        vmax=2.,
    )
    plt.colorbar(m, ax=ax4, label='Fractional Sea Ice (per day)')

    plt.suptitle(f'{date_as_str}')

    plt.tight_layout()
    plt.show()


def infer(f: pathlib.Path, it: int):
    print(f'Using file {f.name}...')
    npz = np.load(f)

    Ni, Mi = (11, 11)
    HXi, HYi = Ni // 2, Mi // 2
    lt, lx, ly = npz['seaice_conc_interp'].shape

    truth = npz['seaice_conc_interp'][it + 1, HXi:lx - HXi, HYi:ly - HYi]

    data = torch.tensor(
        npz['seaice_conc_interp'][it - 2:it + 1][None, ...],  # unsqueeze B dimension
    ).to(dtype=neuralpde.network.DTYPE, device='cuda')
    patch = torch.nn.functional.unfold(data, Ni).reshape((3, Ni, Mi, (lx - 2 * HXi) * (ly - 2 * HYi)))
    patch = patch.moveaxis(-1, 0)

    outputs = torch.full(((lx - 2 * HXi) * (ly - 2 * HYi), 5), 0, dtype=neuralpde.network.DTYPE, device='cuda')
    for i in range(len(patch) // BATCHSIZE + 1):
        outputs[i * BATCHSIZE:(i + 1) * BATCHSIZE] = n.forward(patch[i * BATCHSIZE:(i + 1) * BATCHSIZE])

    patch_mask = torch.tensor(npz['mask'][it, HXi:lx - HXi, HYi:ly - HYi]).to(device='cuda')
    outputs[patch_mask.ravel()] = torch.nan
    outputs = outputs.reshape(((lx - 2 * HXi), (ly - 2 * HYi), 5))

    plot_outputs(
        outputs.detach().cpu().numpy(),
        truth,
        str(npz['date'][it])
    )


def infer_from_logical_names(year: int, day_of_year: int):
    infer(npz_files[year - 1978], day_of_year - 1)


In [ ]:
infer_from_logical_names(2012, 155)
